In [12]:
from adjustText import adjust_text
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset
import ast
import seaborn as sns
from matplotlib.ticker import PercentFormatter
dataset = load_dataset('lukebarousse/data_jobs')
df = dataset['train'].to_pandas()
df['job_posted_date'] = pd.to_datetime(df['job_posted_date'])
df['job_skills'] = df['job_skills'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else x)

In [3]:
df_DA_IND = df[(df['job_title_short'] == 'Data Analyst') & (df['job_country'] == 'India')].copy()

df_DA_IND = df_DA_IND.dropna(subset=['salary_year_avg'])

df_DA_IND_exploded = df_DA_IND.explode('job_skills')

df_DA_IND_exploded[['salary_year_avg', 'job_skills']].head(5)


,salary_year_avg,job_skills
11412,50400.0,python
11412,50400.0,sql
11412,50400.0,aws
11412,50400.0,azure
11412,50400.0,excel


In [4]:
df_DA_skills = df_DA_IND_exploded.groupby('job_skills')['salary_year_avg'].agg(['count', 'median']).sort_values(by='count', ascending=False)

df_DA_skills = df_DA_skills.rename(columns={'count':'skill_count', 'median':'median_salary'})

In [5]:
df_DA_skills

,skill_count,median_salary
job_skills,,
sql,46,96050.0
excel,39,98500.0
python,36,96050.0
tableau,20,108087.5
r,18,79200.0
...,...,...
unify,1,79200.0
unreal,1,105650.0
visio,1,119250.0


In [6]:
DA_job_count = len(df_DA_IND)
DA_job_count

94

In [7]:
df_DA_skills['skill_percent'] = df_DA_skills['skill_count'] / DA_job_count * 100

df_DA_skills

,skill_count,median_salary,skill_percent
job_skills,,,
sql,46,96050.0,48.936170
excel,39,98500.0,41.489362
python,36,96050.0,38.297872
tableau,20,108087.5,21.276596
r,18,79200.0,19.148936
...,...,...,...
unify,1,79200.0,1.063830
unreal,1,105650.0,1.063830
visio,1,119250.0,1.063830


In [8]:
skill_limit = 5

df_DA_skills_high_demand = df_DA_skills[df_DA_skills['skill_percent']>skill_limit]

df_DA_skills_high_demand


,skill_count,median_salary,skill_percent
job_skills,,,
sql,46,96050.00,48.936170
excel,39,98500.00,41.489362
python,36,96050.00,38.297872
tableau,20,108087.50,21.276596
r,18,79200.00,19.148936
power bi,17,111175.00,18.085106
azure,15,93600.00,15.957447
aws,12,79200.00,12.765957
oracle,11,79200.00,11.702128


In [ ]:
plot_df = df_DA_skills_high_demand.copy()
_seen = {}
_dx = 0.45
for _i in range(len(plot_df)):
    _key = (plot_df['skill_percent'].iloc[_i], plot_df['median_salary'].iloc[_i])
    if _key in _seen:
        _seen[_key] += 1
        plot_df.iloc[_i, plot_df.columns.get_loc('skill_percent')] = (
            plot_df['skill_percent'].iloc[_i] + _dx * _seen[_key]
        )
    else:
        _seen[_key] = 0


plt.scatter(plot_df['skill_percent'], plot_df['median_salary'])
texts = []
for i, txt in enumerate(plot_df.index):
    texts.append(plt.text(plot_df['skill_percent'].iloc[i], plot_df['median_salary'].iloc[i], txt))

adjust_text(
    texts,
    x=plot_df['skill_percent'].values,
    y=plot_df['median_salary'].values,
    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8, shrinkA=2, shrinkB=3),
    iter_lim=2000,
    force_text=(0.25, 0.5),
    force_static=(0.2, 0.4),
    force_explode=(0.15, 0.35),
    expand=(1.2, 1.4),
)

plt.xlabel('Percent of Job Postings')
plt.ylabel('Median Yearly Salary')
plt.title('Most Optimal Skills for Data Analysts in India')


ax = plt.gca()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, pos: f'${int(y / 1000)}K'))
ax.xaxis.set_major_formatter(PercentFormatter(decimals=0))


plt.tight_layout()
plt.show()


In [15]:
df['job_type_skills'].head(10)

0                                                 None
1    {'analyst_tools': ['power bi', 'tableau'], 'pr...
2    {'analyst_tools': ['dax'], 'cloud': ['azure'],...
3    {'cloud': ['aws'], 'libraries': ['tensorflow',...
4    {'cloud': ['oracle', 'aws'], 'other': ['ansibl...
5    {'cloud': ['gcp'], 'programming': ['python', '...
6    {'cloud': ['gcp', 'bigquery'], 'databases': ['...
7    {'cloud': ['gcp', 'azure', 'aws', 'bigquery', ...
8    {'analyst_tools': ['excel', 'powerpoint', 'pow...
9    {'analyst_tools': ['excel'], 'cloud': ['azure'...
Name: job_type_skills, dtype: object

In [17]:
df_tech = df['job_type_skills'].copy()

df_tech = df_tech.drop_duplicates()

df_tech = df_tech.dropna()

tech_dict = {}
for row in df_tech:
    row_dict = ast.literal_eval(row)
    for key, value in row_dict.items():
        if key in tech_dict:
            tech_dict[key] += value
        else:
            tech_dict[key] = value

    for key, value in tech_dict.items():
        tech_dict[key] = list(set(value))


            

In [18]:
df_tech = pd.DataFrame(list(tech_dict.items()), columns =['technology', 'skills'])

df_tech = df_tech.explode('skills')

df_tech

,technology,skills
0,analyst_tools,sheets
0,analyst_tools,word
0,analyst_tools,sharepoint
0,analyst_tools,esquisse
0,analyst_tools,msaccess
...,...,...
9,sync,twilio
9,sync,wire
9,sync,slack
9,sync,google chat


In [28]:
df_plot = df_DA_skills_high_demand.merge(df_tech, left_on='job_skills', right_on='skills')
df_plot = df_plot.drop_duplicates(subset=['skills'])
df_plot

,skill_count,median_salary,skill_percent,technology,skills
0,46,96050.0,48.93617,programming,sql
1,39,98500.0,41.489362,analyst_tools,excel
2,36,96050.0,38.297872,programming,python
3,20,108087.5,21.276596,analyst_tools,tableau
4,18,79200.0,19.148936,programming,r
5,17,111175.0,18.085106,analyst_tools,power bi
6,15,93600.0,15.957447,cloud,azure
7,12,79200.0,12.765957,cloud,aws
8,11,79200.0,11.702128,cloud,oracle
9,11,111175.0,11.702128,libraries,spark


In [ ]:
sns.scatterplot(
    data=df_plot,
    x='skill_percent',
    y='median_salary',
    hue='technology'
    
)

sns.set_theme(style='ticks')
sns.despine()
texts = []
for i, txt in enumerate(df_plot['skills']):
    texts.append(plt.text(df_plot['skill_percent'].iloc[i], df_plot['median_salary'].iloc[i], txt))

adjust_text(
    texts,
    x=df_plot['skill_percent'].values,
    y=df_plot['median_salary'].values,
    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8, shrinkA=2, shrinkB=3),
    iter_lim=2000,
    force_text=(0.25, 0.5),
    force_static=(0.2, 0.4),
    force_explode=(0.15, 0.35),
    expand=(1.2, 1.4),
)

plt.xlabel('Percent of Job Postings')
plt.ylabel('Median Yearly Salary')
plt.title('Most Optimal Skills for Data Analysts in India')


ax = plt.gca()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, pos: f'${int(y / 1000)}K'))
ax.xaxis.set_major_formatter(PercentFormatter(decimals=0))


plt.tight_layout()
plt.show()